# YOLOv8n Chessboard Detection - Training & Evaluation

Train a YOLOv8n model to detect chessboards in GothamChess video frames.
Single class: `chessboard`.

In [1]:
from pathlib import Path
import yaml

DATASET_YAML = Path("./data/yolo_dataset/dataset.yaml")

with open(DATASET_YAML) as f:
    ds_config = yaml.safe_load(f)
print("Dataset config:", ds_config)

# Verify counts
for split in ["train", "val", "test"]:
    img_dir = Path(ds_config["path"]) / ds_config[split]
    n = len(list(img_dir.glob("*.png")))
    print(f"  {split}: {n} images")

Dataset config: {'path': '/home/amicus/Desktop/ChessCVModel/board_detection/data/yolo_dataset', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'chessboard'}}
  train: 1047 images
  val: 185 images
  test: 296 images


## Train YOLOv8n

In [2]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATASET_YAML.resolve()),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    flipud=0.0,
    fliplr=0.3,
    mosaic=0.5,
    mixup=0.1,
    # Output
    project="runs",
    name="yolov8n_chessboard",
    device=0,
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16045MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/amicus/Desktop/ChessCVModel/board_detection/data/yolo_dataset/dataset.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.3, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov8n_chessboard, nbs=64, nms=False, opset=N

## Evaluate on Test Set

In [5]:
best_model = YOLO("./runs/detect/runs/yolov8n_chessboard/weights/best.pt")

metrics = best_model.val(
    data=str(DATASET_YAML.resolve()),
    split="test",
)

print(f"mAP@0.5:     {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16045MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7859.5±1204.8 MB/s, size: 588.1 KB)
val: Scanning /home/amicus/Desktop/ChessCVModel/board_detection/data/yolo_dataset/labels/test... 296 images, 24 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 296/296 1.1Kit/s 0.3s<0.2s
val: New cache created: /home/amicus/Desktop/ChessCVModel/board_detection/data/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 17.9it/s 1.1s0.1s
                   all        296        272          1          1      0.995      0.995
Speed: 0.9ms preprocess, 0.7ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/amicus/Desktop/ChessCVModel/board_detection/runs/detect/val
mAP@0.5:     0.9950
mAP@0.5:0.95: 0.9950
Precision:    0

## Visual Inspection

In [7]:
import cv2
import matplotlib.pyplot as plt
import random

test_images = sorted((Path(ds_config["path"]) / ds_config["test"]).glob("*.png"))
samples = random.sample(test_images, min(12, len(test_images)))

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax, img_path in zip(axes.flat, samples):
    results = best_model.predict(str(img_path), verbose=False)
    annotated = results[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.stem[:30], fontsize=8)
    ax.axis("off")

for ax in axes.flat[len(samples):]:
    ax.axis("off")

plt.suptitle("YOLOv8n Chessboard Detection - Test Predictions", fontsize=14)
plt.tight_layout()
plt.savefig("./runs/detect/runs/yolov8n_chessboard/test_predictions.png", dpi=150)
plt.show()

<Figure size 2000x1500 with 12 Axes>

<Figure size 2000x1500 with 12 Axes>

## Export to ONNX

In [8]:
best_model.export(format="onnx", imgsz=640, simplify=True)
print("Exported to ONNX")

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0 CPU (Intel Core i7-14700K)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'runs/detect/runs/yolov8n_chessboard/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/252.8 MB ? eta -:--:--
   ━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/252.8 MB 101.3 MB/s eta 0:00:03
   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/252.8 MB 108.6 MB/s eta 0:00:02
   ━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/252.8 MB 111.3 MB/s eta 0:00:02
   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/252.8 MB 111.6 MB/s eta 0:00:02
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 104.9/252.8 MB 104.4 MB/s eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━

/home/amicus/miniconda3/envs/chesscv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(
2026-03-05 14:22:55.161437767 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


ONNX: export success ✅ 5.4s, saved as 'runs/detect/runs/yolov8n_chessboard/weights/best.onnx' (11.7 MB)

Export complete (5.5s)
Results saved to /home/amicus/Desktop/ChessCVModel/board_detection/runs/detect/runs/yolov8n_chessboard/weights
Predict:         yolo predict task=detect model=runs/detect/runs/yolov8n_chessboard/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs/detect/runs/yolov8n_chessboard/weights/best.onnx imgsz=640 data=/home/amicus/Desktop/ChessCVModel/board_detection/data/yolo_dataset/dataset.yaml  
Visualize:       https://netron.app
Exported to ONNX


## Compare: YOLO vs Template Matching

Run both detectors on ChessAtlasBackend test images to compare.

In [9]:
import sys
sys.path.insert(0, str(Path(".").resolve()))
from pathlib import Path

backend_test_dir = Path.home() / "Documents/GitHub/ChessAtlasBackend/test_images"
backend_frames = sorted(backend_test_dir.glob("*.png"))[:10]

print(f"Comparing on {len(backend_frames)} backend test images\n")

for img_path in backend_frames:
    results = best_model.predict(str(img_path), verbose=False)
    boxes = results[0].boxes
    if len(boxes) > 0:
        conf = boxes.conf[0].item()
        print(f"  {img_path.name}: YOLO detected (conf={conf:.3f})")
    else:
        print(f"  {img_path.name}: YOLO no detection")

Comparing on 0 backend test images

